# Data Processing
Climate data loading, preprocessing, feature engineering, and train/val/test split.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings, glob
warnings.filterwarnings('ignore')

# Global constants
EXTREME_T   = 28        # Extreme heat threshold (°C)
MAN_LAT     = 53.48
MAN_LON_360 = 360 - 2.24  # Manchester longitude in 0-360 format


In [ ]:
# ─── Global plot settings (journal style) ───────────────────────────────────
matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  9,
    'figure.dpi':       130,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.22,
    'grid.linestyle':   '--',
    'axes.axisbelow':   True,
})

C = dict(
    RF='#2166AC', XGB='#4DAC26', LGB='#E08214',
    LSTM='#762A83', CNN='#D6604D', CNN_LSTM='#01665E',
    Ensemble='#543005', obs='#252525',
    hist='#4393C3', fut='#D6604D', extreme='#B2182B',
)

def savefig(name):
    plt.savefig(f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')


In [140]:
# =============================================================================
# PART 1  数据加载
# =============================================================================
import numpy as np
import pandas as pd
import xarray as xr
import glob, warnings
warnings.filterwarnings('ignore')
 
# ─── 全局设置 ────────────────────────────────────────────────────────────────
EXTREME_T   = 28        # 极端高温阈值 (°C)
MAN_LAT     = 53.48     # 曼彻斯特纬度
MAN_LON_360 = 360 - 2.24   # 你的数据经度是 0-360 格式，-2.24° → 357.76°
 
print('=' * 65)
print('PART 1  —  Data Loading & Preprocessing (Climate Calendar Fix)')
print('=' * 65)
 
# =============================================================================
# STEP 1: 加载数据
# =============================================================================
 
print('\n[Step 1] Loading datasets...')
 
DATA_PATH = '/Users/william/Downloads/project_2/*.nc'   # ← 你的实际路径
 
try:
    # 优先使用已加载的 ds（避免重复读取）
    _ = ds
    print(f'  Using existing `ds` variable in memory')
    print(f'  Dimensions: {dict(ds.dims)}')
except NameError:
    # 重新加载
    files = sorted(glob.glob(DATA_PATH))
    if not files:
        raise FileNotFoundError(f'No .nc files found at: {DATA_PATH}')
    print(f'  Found {len(files)} files, concatenating...')
    datasets = [xr.open_dataset(f) for f in files]
    ds = xr.concat(datasets, dim='time')
    print(f'  Concatenated shape: {dict(ds.dims)}')
 
print(f'  Variables: {list(ds.data_vars)}')
 
# =============================================================================
# STEP 2: 提取曼彻斯特最近格点（FIX-C）
# =============================================================================
print('\n[Step 2] Extracting Manchester grid point...')
 
lat_arr = ds['lat'].values
lon_arr = ds['lon'].values   # 0-360 格式
 
# 找最近格点
lat_idx = int(np.argmin(np.abs(lat_arr - MAN_LAT)))
lon_idx = int(np.argmin(np.abs(lon_arr - MAN_LON_360)))
 
print(f'  Target:  lat={MAN_LAT}, lon={MAN_LON_360:.2f} (= -2.24° in 0-360)')
print(f'  Nearest: lat={lat_arr[lat_idx]:.2f}, lon={lon_arr[lon_idx]:.2f}')
 
# 提取该格点的所有变量
ds_man = ds.isel(lat=lat_idx, lon=lon_idx)
print(f'  Time steps: {len(ds_man.time)}')
 
# 检查 TREFMXAV_U 的 NaN 情况
trefmx_arr = ds_man['TREFMXAV_U'].values
nan_frac = np.isnan(trefmx_arr).mean()
print(f'  TREFMXAV_U NaN fraction: {nan_frac:.1%}')
 
if nan_frac > 0.5:
    print('  ⚠ TREFMXAV_U has many NaNs at this point.')
    print('  → Trying nearby lat/lon points...')
    # 尝试周围格点
    found_valid = False
    for dlat in range(-2, 3):
        for dlon in range(-2, 3):
            li = lat_idx + dlat
            lo = lon_idx + dlon
            if 0 <= li < len(lat_arr) and 0 <= lo < len(lon_arr):
                test = ds.isel(lat=li, lon=lo)['TREFMXAV_U'].values
                if np.isnan(test).mean() < 0.3:
                    lat_idx, lon_idx = li, lo
                    ds_man = ds.isel(lat=lat_idx, lon=lon_idx)
                    print(f'  → Found valid point: lat={lat_arr[lat_idx]:.2f}, '
                          f'lon={lon_arr[lon_idx]:.2f}  '
                          f'(NaN={np.isnan(ds_man["TREFMXAV_U"].values).mean():.1%})')
                    found_valid = True
                    break
        if found_valid:
            break
    if not found_valid:
        print('  → No urban point with TREFMXAV_U found.')
        print('  → Falling back to TREFHT (non-urban max temperature)')
 
# =============================================================================
# STEP 3: 时间轴转换（FIX-B：正确处理 noleap / 360_day）
# =============================================================================
print('\n[Step 3] Converting climate calendar time axis...')
 
import cftime
 
time_raw = ds_man['time'].values
 
def convert_cftime_to_ordinal(time_arr):
    """
    将 cftime 数组转为 pandas-compatible 日期序列。
    保持气候时间精度，不强制转 datetime64（避免 noleap/360_day 报错）。
    返回：
        dates_str  : list of 'YYYY-MM-DD' 字符串（用于显示）
        year_arr   : np.array of int (年份)
        month_arr  : np.array of int (月份)
        day_arr    : np.array of int (日)
        doy_arr    : np.array of int (day of year, 1-based)
    """
    years, months, days, doys = [], [], [], []
 
    for t in time_arr:
        if isinstance(t, (cftime.DatetimeNoLeap,
                          cftime.Datetime360Day,
                          cftime.DatetimeProlepticGregorian,
                          cftime.DatetimeGregorian,
                          cftime.DatetimeAllLeap)):
            y, m, d = t.year, t.month, t.day
            # Day of year：手动计算（兼容 360_day）
            days_in_months_noleap = [31,28,31,30,31,30,31,31,30,31,30,31]
            days_in_months_360    = [30]*12
            if isinstance(t, cftime.Datetime360Day):
                doy = sum(days_in_months_360[:m-1]) + d
            else:
                doy = sum(days_in_months_noleap[:m-1]) + d
        else:
            # 标准 datetime / numpy datetime64
            try:
                ts = pd.Timestamp(str(t))
                y, m, d = ts.year, ts.month, ts.day
                doy = ts.dayofyear
            except Exception:
                y, m, d, doy = 2006, 1, 1, 1
 
        years.append(y); months.append(m)
        days.append(d);  doys.append(doy)
 
    return (np.array(years), np.array(months),
            np.array(days),  np.array(doys))
 
year_arr, month_arr, day_arr, doy_arr = convert_cftime_to_ordinal(time_raw)
 
print(f'  Calendar type: {type(time_raw[0]).__name__}')
print(f'  Year range:    {year_arr.min()} – {year_arr.max()}')
print(f'  Total steps:   {len(year_arr)}')
 
# =============================================================================
# STEP 4: 构建主 DataFrame（多变量）
# =============================================================================
print('\n[Step 4] Building multi-variable DataFrame...')
 
def safe_get(ds_pt, var_name, unit_conv=None):
    """安全提取变量，若不存在返回 NaN 数组"""
    if var_name in ds_pt.data_vars:
        arr = ds_pt[var_name].values.astype(float)
        if unit_conv == 'K2C':
            arr = arr - 273.15
        elif unit_conv == 'ms2mmd':   # m/s → mm/day
            arr = arr * 86400 * 1000
        return arr
    else:
        return np.full(len(ds_pt.time), np.nan)
 
# TREFMXAV_U: 城市最高气温，优先使用；大量NaN时改用TREFHT
trefmxav = safe_get(ds_man, 'TREFMXAV_U')
trefmxav_C = trefmxav - 273.15 if np.nanmax(trefmxav) > 200 else trefmxav
 
# TREFHT: 参考高度温度（用于辅助特征）
trefht   = safe_get(ds_man, 'TREFHT')
trefht_C = trefht - 273.15 if np.nanmax(trefht) > 200 else trefht
 
# 选定主要预测目标：优先 TREFMXAV_U，否则用 TREFHT
nan_trefmx = np.isnan(trefmxav_C).mean()
if nan_trefmx > 0.5:
    print(f'  ⚠ TREFMXAV_U NaN={nan_trefmx:.0%} → using TREFHT as main target')
    temperature = trefht_C
    target_name = 'TREFHT'
else:
    temperature = trefmxav_C
    target_name = 'TREFMXAV_U'
print(f'  Target variable: {target_name}')
print(f'  Temp range: {np.nanmin(temperature):.1f} – {np.nanmax(temperature):.1f}°C')
 
# 构建 DataFrame
df_raw = pd.DataFrame({
    'year':        year_arr,
    'month':       month_arr,
    'day':         day_arr,
    'doy':         doy_arr,
    'temperature': temperature,
    'TREFHT_C':    trefht_C,
    'FSNS':        safe_get(ds_man, 'FSNS'),
    'FLNS':        safe_get(ds_man, 'FLNS'),
    'PRECT':       safe_get(ds_man, 'PRECT', 'ms2mmd'),  # mm/day
    'PRSN':        safe_get(ds_man, 'PRSN',  'ms2mmd'),
    'QBOT':        safe_get(ds_man, 'QBOT'),
    'UBOT':        safe_get(ds_man, 'UBOT'),
    'VBOT':        safe_get(ds_man, 'VBOT'),
})
 
# 添加 ensemble_member 列（单一合并数据集，标为 0）

df_raw['ensemble_member'] = 0
 
print(f'  DataFrame shape: {df_raw.shape}')
print(f'  NaN summary:\n{df_raw.isnull().sum()}')
 
# 删除 temperature 全为 NaN 的行
df_raw = df_raw.dropna(subset=['temperature']).reset_index(drop=True)
print(f'  After dropna(temperature): {len(df_raw)} rows')

PART 1  —  Data Loading & Preprocessing (Climate Calendar Fix)

[Step 1] Loading datasets...
  Using existing `ds` variable in memory
  Dimensions: {'time': 27374, 'lat': 11, 'lon': 6}
  Variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']

[Step 2] Extracting Manchester grid point...
  Target:  lat=53.48, lon=357.76 (= -2.24° in 0-360)
  Nearest: lat=53.25, lon=357.50
  Time steps: 27374
  TREFMXAV_U NaN fraction: 0.0%

[Step 3] Converting climate calendar time axis...
  Calendar type: DatetimeNoLeap
  Year range:    2006 – 2080
  Total steps:   27374

[Step 4] Building multi-variable DataFrame...
  Target variable: TREFMXAV_U
  Temp range: -0.2 – 36.8°C
  DataFrame shape: (27374, 14)
  NaN summary:
year               0
month              0
day                0
doy                0
temperature        0
TREFHT_C           0
FSNS               0
FLNS               0
PRECT              0
PRSN               0
QBOT               0
UBOT               

In [141]:
# =============================================================================
# PART 2  特征工程
# =============================================================================
print('\n[Step 5] Feature engineering (no target leakage)...')
 
def build_features(df):
    d = df.copy()
 
    # 1. 时间循环编码
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)
    d['doy_sin']   = np.sin(2 * np.pi * d['doy'] / 365)
    d['doy_cos']   = np.cos(2 * np.pi * d['doy'] / 365)
 
    # 2. 季节独热
    d['is_summer'] = d['month'].isin([6,7,8]).astype(int)
    d['is_winter'] = d['month'].isin([12,1,2]).astype(int)
    d['is_spring'] = d['month'].isin([3,4,5]).astype(int)
    d['is_autumn'] = d['month'].isin([9,10,11]).astype(int)
 
    # 3. 长期趋势
    yr_min, yr_max = d['year'].min(), d['year'].max()
    d['year_norm'] = (d['year'] - yr_min) / max(yr_max - yr_min, 1)
 
    # 4. 滞后特征（shift(1) = t-1，不含当天）
    g = d.groupby('ensemble_member')['temperature']
    for lag in [1, 2, 3, 7, 14, 30]:
        d[f'temp_lag_{lag}'] = g.shift(lag)
 
    # 5. 滚动统计（shift(1) 后滚动，不包含当天）
    for w in [7, 14, 30]:
        d[f'roll_mean_{w}'] = g.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        d[f'roll_std_{w}']  = g.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).std())
        d[f'roll_max_{w}']  = g.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).max())
        d[f'roll_min_{w}']  = g.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).min())
 
    # 6. 差分特征（温度变化速率）
    for lag in [1, 7, 30]:
        d[f'temp_diff_{lag}'] = g.shift(1) - g.shift(lag + 1)
 
    # 7. 气候变量的滞后（TREFHT, FSNS, PRECT 是好的物理预测因子）
    for var in ['TREFHT_C', 'FSNS', 'PRECT', 'QBOT']:
        if var in d.columns:
            gv = d.groupby('ensemble_member')[var]
            d[f'{var}_lag1'] = gv.shift(1)
            d[f'{var}_lag7'] = gv.shift(7)
 
    return d
 
data_featured = build_features(df_raw)
 
# 极端标志（用原始°C）
data_featured['is_extreme'] = (data_featured['temperature'] > EXTREME_T).astype(int)
 
# 定义特征列（严格排除目标变量）
META_COLS = ['year','month','day','doy','temperature','ensemble_member',
             'is_extreme','TREFMXAV_U']
feature_columns = [c for c in data_featured.columns if c not in META_COLS]
 
# 删除 NaN（由 lag/rolling 产生）
data_featured = data_featured.dropna(subset=feature_columns + ['temperature'])
data_featured = data_featured.reset_index(drop=True)
 
print(f'  Features ({len(feature_columns)}):')
for i, f in enumerate(feature_columns):
    print(f'    {i+1:2d}. {f}')
print(f'\n  Final dataset: {data_featured.shape}')
print(f'  Extreme heat days: {data_featured["is_extreme"].sum():,} '
      f'({data_featured["is_extreme"].mean()*100:.1f}%)')
 
# =============================================================================
# PART 3: 时序切分（不shuffle，按年份）
# =============================================================================
print('\n[Step 6] Temporal train/val/test split...')
 
year_all = data_featured['year'].values
total_years = year_all.max() - year_all.min() + 1
# 约 70% / 15% / 15%
cutoff_train = year_all.min() + int(total_years * 0.70)
cutoff_val   = year_all.min() + int(total_years * 0.85)
 
print(f'  Split: train ≤{cutoff_train}, val {cutoff_train+1}–{cutoff_val}, test >{cutoff_val}')
 
mask_tr  = year_all <= cutoff_train
mask_val = (year_all > cutoff_train) & (year_all <= cutoff_val)
mask_te  = year_all > cutoff_val
 
df_train = data_featured[mask_tr].copy()
df_val   = data_featured[mask_val].copy()
df_test  = data_featured[mask_te].copy()
 
print(f'  Train: {df_train["year"].min()}–{df_train["year"].max()} | {len(df_train):>7,} rows')
print(f'  Val  : {df_val["year"].min()}–{df_val["year"].max()}   | {len(df_val):>7,} rows')
print(f'  Test : {df_test["year"].min()}–{df_test["year"].max()}   | {len(df_test):>7,} rows')
 
# 标准化（只在训练集上 fit）
from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler().fit(df_train[feature_columns])
scaler_y = StandardScaler().fit(df_train[['temperature']])
 
def scale_X(df):  return scaler_X.transform(df[feature_columns])
def scale_y(df):  return scaler_y.transform(df[['temperature']]).ravel()
def inv_y(arr):   return scaler_y.inverse_transform(
    np.array(arr).reshape(-1,1)).ravel()
 
X_train_s = scale_X(df_train); y_train_s = scale_y(df_train)
X_val_s   = scale_X(df_val);   y_val_s   = scale_y(df_val)
X_test_s  = scale_X(df_test);  y_test_s  = scale_y(df_test)
 
y_train_C = df_train['temperature'].values
y_val_C   = df_val['temperature'].values
y_test_C  = df_test['temperature'].values
 
print(f'\n  Test temperature: {y_test_C.min():.1f} – {y_test_C.max():.1f}°C')
print('\n✅ Part 1 complete. Variables ready for Part 2 (models).')
print(f'   data_featured, feature_columns, df_train/val/test')
print(f'   X/y train/val/test scaled, scaler_X, scaler_y, inv_y()')
 


[Step 5] Feature engineering (no target leakage)...
  Features (46):
     1. TREFHT_C
     2. FSNS
     3. FLNS
     4. PRECT
     5. PRSN
     6. QBOT
     7. UBOT
     8. VBOT
     9. month_sin
    10. month_cos
    11. doy_sin
    12. doy_cos
    13. is_summer
    14. is_winter
    15. is_spring
    16. is_autumn
    17. year_norm
    18. temp_lag_1
    19. temp_lag_2
    20. temp_lag_3
    21. temp_lag_7
    22. temp_lag_14
    23. temp_lag_30
    24. roll_mean_7
    25. roll_std_7
    26. roll_max_7
    27. roll_min_7
    28. roll_mean_14
    29. roll_std_14
    30. roll_max_14
    31. roll_min_14
    32. roll_mean_30
    33. roll_std_30
    34. roll_max_30
    35. roll_min_30
    36. temp_diff_1
    37. temp_diff_7
    38. temp_diff_30
    39. TREFHT_C_lag1
    40. TREFHT_C_lag7
    41. FSNS_lag1
    42. FSNS_lag7
    43. PRECT_lag1
    44. PRECT_lag7
    45. QBOT_lag1
    46. QBOT_lag7

  Final dataset: (27343, 53)
  Extreme heat days: 203 (0.7%)

[Step 6] Temporal train/val/te

In [143]:
# =============================================================================
# PART 4  评估工具函数（FIX-2: 全部用°C）
# =============================================================================
 
def evaluate_C(y_true_C, y_pred_C, name=''):
    """统一用原始°C计算所有指标，杜绝scaled/原始混用"""
    mae  = mean_absolute_error(y_true_C, y_pred_C)
    rmse = np.sqrt(mean_squared_error(y_true_C, y_pred_C))
    r2   = r2_score(y_true_C, y_pred_C)
    mape = np.mean(np.abs((y_true_C - y_pred_C) / (np.abs(y_true_C) + 1e-6))) * 100
    if name:
        print(f'  {name:<18} MAE={mae:.3f}°C  RMSE={rmse:.3f}°C  '
              f'R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}
 
def extreme_scores(y_true_C, y_pred_C, thr=EXTREME_T):
    """FIX-6: 用原始°C做阈值判断"""
    obs  = (y_true_C > thr).astype(int)
    pred = (y_pred_C > thr).astype(int)
    tp = int(((obs==1)&(pred==1)).sum())
    fp = int(((obs==0)&(pred==1)).sum())
    fn = int(((obs==1)&(pred==0)).sum())
    pod = tp/(tp+fn) if (tp+fn)>0 else 0.0
    far = fp/(tp+fp) if (tp+fp)>0 else 0.0
    ts  = tp/(tp+fp+fn) if (tp+fp+fn)>0 else 0.0
    return {'POD':pod,'FAR':far,'TS':ts,'TP':tp,'FP':fp,'FN':fn}